# Text Modalities and Stress in Text

**By NMA NLP Project Mawu-lise Team**

__Project Members:__  Pin-Chun Chen, Liam Hart, Victor Martins, Hamid Abuwarda, Esteban Leon

---
# Objectives
We're interested in automatically classifying text. There is a large dataset (Hippocorpus) with various modalities of text (recollected, summarized, imaginative, and remembered) at different levels of self-report stress (1 to 5 on a Likert scale). We will use a subset of this data, specifically the estimated frequency of words, to conduct a pilot study investigating whether we can classify different modalities and stress levels based on vocabulary frequency. Then, we will test the size of the improvement that adding semantics as a predictor improves classification accuracy. And if so, which segments (beginning, middle, and ending) of the text (if not all) are necessary for good decoding performance?

Please check out the different resources below to better understand the Hippocorpus dataset and learn more about the texts.

**Resources**:
* [See Hipocorpus paper here](https://www.pnas.org/doi/epdf/10.1073/pnas.2211715119)
* [Hippocorpus dataset](https://huggingface.co/datasets/allenai/hippocorpus)
* [Our GitHub page](https://github.com/pinchunc/NMA_DL_SentimentAnalysis/tree/main)

---
# Setup


In [ ]:
# @title Import libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats

from collections import Counter, defaultdict
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import spacy

import re
from sklearn.utils import shuffle
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from textblob import TextBlob          # or vaderSentiment / spaCy / transformers

from functools import partial
import nltk, warnings
from nltk import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer

from IPython.display import display
from datetime import datetime
import random

# now_utc = datetime.utcnow()
# print(now_utc)  # e.g., 2025-07-11 11:47:00.123456


In [ ]:
# @title Set random seed
# Call `set_seed` function to ensure reproducibility.

def set_seed(seed=None, seed_torch=True):
  """
  Function that controls randomness. NumPy and random modules must be imported.

  Args:
    seed : Integer
      A non-negative integer that defines the random state. Default is `None`.
    seed_torch : Boolean
      If `True` sets the random seed for pytorch tensors, so pytorch module
      must be imported. Default is `True`.

  Returns:
    Nothing.
  """
  if seed is None:
    seed = np.random.choice(2 ** 32)
  random.seed(seed)
  np.random.seed(seed)
  if seed_torch:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

  print(f'Random seed {seed} has been set.')


# In case that `DataLoader` is used
def seed_worker(worker_id):
  """
  DataLoader will reseed workers following randomness in
  multi-process data loading algorithm.

  Args:
    worker_id: integer
      ID of subprocess to seed. 0 means that
      the data will be loaded in the main process
      Refer: https://pytorch.org/docs/stable/data.html#data-loading-randomness for more details

  Returns:
    Nothing
  """
  worker_seed = torch.initial_seed() % 2**32
  np.random.seed(worker_seed)
  random.seed(worker_seed)

SEED = 2025
set_seed(seed=SEED)

In [ ]:
# @title Set device (GPU or CPU).
# Inform the user if the notebook uses GPU or CPU.

def set_device():
  """
  Set the device. CUDA if available, CPU otherwise

  Args:
    None

  Returns:
    Nothing
  """
  device = "cuda" if torch.cuda.is_available() else "cpu"
  if device != "cuda":
    print("WARNING: For this notebook to perform best, "
        "if possible, in the menu under `Runtime` -> "
        "`Change runtime type.`  select `GPU` ")
  else:
    print("GPU is enabled in this notebook.")

  return device

DEVICE = set_device()

In [ ]:
# @title Helper Functions from Twiter Sentiment Analysis NMA Colab
_patterns = [r"\'", r"\"", r"\.", r"<br \/>", r",", r"\(", r"\)", r"\!", r"\?", r"\;", r"\:", r"\s+"]

_replacements = [" '  ", "", " . ", " ", " , ", " ( ", " ) ", " ! ", " ? ", " ", " ", " "]

_patterns_dict = list((re.compile(p), r) for p, r in zip(_patterns, _replacements))

def _basic_english_normalize(line):
    r"""
    Basic normalization for a line of text.
    Normalization includes
    - lowercasing
    - complete some basic text normalization for English words as follows:
        add spaces before and after '\''
        remove '\"',
        add spaces before and after '.'
        replace '<br \/>'with single space
        add spaces before and after ','
        add spaces before and after '('
        add spaces before and after ')'
        add spaces before and after '!'
        add spaces before and after '?'
        replace ';' with single space
        replace ':' with single space
        replace multiple spaces with single space

    Returns a list of tokens after splitting on whitespace.
    """

    line = line.lower()
    for pattern_re, replaced_str in _patterns_dict:
        line = pattern_re.sub(replaced_str, line)
    return line.split()



def get_tokenizer(tokenizer, language='en'):
    r"""
    Generate tokenizer function for a string sentence.

    Arguments:
        tokenizer: the name of tokenizer function. If None, it returns split()
            function, which splits the string sentence by space.
            If basic_english, it returns _basic_english_normalize() function,
            which normalize the string first and split by space. If a callable
            function, it will return the function. If a tokenizer library
            (e.g. spacy, moses, toktok, revtok, subword), it returns the
            corresponding library.
        language: Default en

    Examples:
        >>> import torchtext
        >>> from torchtext.data import get_tokenizer
        >>> tokenizer = get_tokenizer("basic_english")
        >>> tokens = tokenizer("You can now install TorchText using pip!")
        >>> tokens
        >>> ['you', 'can', 'now', 'install', 'torchtext', 'using', 'pip', '!']

    """

    # default tokenizer is string.split(), added as a module function for serialization


    if tokenizer == "basic_english":
        if language != 'en':
            raise ValueError("Basic normalization is only available for Enlish(en)")
        return _basic_english_normalize

    # simply return if a function is passed
    if callable(tokenizer):
        return tokenizer

    if tokenizer == "spacy":
        try:
            import spacy
            spacy = spacy.load(language)
            return partial(_spacy_tokenize, spacy=spacy)
        except ImportError:
            print("Please install SpaCy. "
                  "See the docs at https://spacy.io for more information.")
            raise
        except AttributeError:
            print("Please install SpaCy and the SpaCy {} tokenizer. "
                  "See the docs at https://spacy.io for more "
                  "information.".format(language))
            raise
    elif tokenizer == "moses":
        try:
            from sacremoses import MosesTokenizer
            moses_tokenizer = MosesTokenizer()
            return moses_tokenizer.tokenize
        except ImportError:
            print("Please install SacreMoses. "
                  "See the docs at https://github.com/alvations/sacremoses "
                  "for more information.")
            raise
    elif tokenizer == "toktok":
        try:
            from nltk.tokenize.toktok import ToktokTokenizer
            toktok = ToktokTokenizer()
            return toktok.tokenize
        except ImportError:
            print("Please install NLTK. "
                  "See the docs at https://nltk.org  for more information.")
            raise
    elif tokenizer == 'revtok':
        try:
            import revtok
            return revtok.tokenize
        except ImportError:
            print("Please install revtok.")
            raise
    elif tokenizer == 'subword':
        try:
            import revtok
            return partial(revtok.tokenize, decap=True)
        except ImportError:
            print("Please install revtok.")
            raise
    raise ValueError("Requested tokenizer {}, valid choices are a "
                     "callable that takes a single string as input, "
                     "\"revtok\" for the revtok reversible tokenizer, "
                     "\"subword\" for the revtok caps-aware tokenizer, "
                     "\"spacy\" for the SpaCy English tokenizer, or "
                     "\"moses\" for the NLTK port of the Moses tokenization "
                     "script.".format(tokenizer))

---
# Step 1: Question

There are many different questions we could ask with the Hippocorpus dataset. We will start with a simple question: **"Can we classify text from frequency of words data?"**

Our goal is to perform a pilot study to see if this is possible in principle. If this works out, then **as a next step we might want to add semantic data or even use only portions of the text...**

The ultimate goal is to figure out which text features to pay attention to to classify texts.

**Objectively:**
* Can we predict the stress level of writing the story based on how the person told the story?
* Can we predict the type of recollection (recalled, retold, or imagined) that the story represents?
* Does the time since the event change the level of stress of writing about it?

---
# Step 2: Literature review

Most importantly, our literature review needs to address the following:
* What modeling approaches make it possible to classify text data?
* How is human emotion captured in words and the way of writing?
* What exactly is in the Hippocorpus dataset?
* What is known regarding the classification of human stress level based on the words used and their distribution in-text?

What we learn from the literature review is too long to write out here... But we would like to point out that human emotion classification has been done; we're not proposing a very novel project here. But that's ok for an NMA project!


---
# Step 3: Ingredients

## Data ingredients

After downloading the data, we should have a dataframe with:

* **IDs:** AssignmentId, WorkerId, recAgnPairId, recImgPairId
* **Likert scale self reported measurements:** distracted, draining, frequency, importance, openness, stressful, similarity
* **Demographics:** annotatorAge, annotatorGender, annotatorRace
* **Narative related text:** mainEvent, memType, mostSurprising, similarityReason, story, summary
* **Metadata:**  WorkTimeInSeconds, logTimeSinceEvent, timeSinceEvent

We'll take a closer look at the data below. 

## Modeling ingredients

* How are words related to emotions? How is a sequece of words related to semantics? Whats tools can we use to figure these out?

* What tools are capable of representing/understanding meaning/semantics?

* How should text be preprossessed to answers the previous questions?

* Classifier? --> To be decided. One dimension of choise is weather a supervised or unsupervised technique would be more appropriate.

**Approach 1:**
* Preprocessing: Bag of words, TF-IDF
  * Advantage: simple, fast, out-of-the-box
* Classification (one of): Naive Bayes, Logistic Regression, SVMs

**Approach 2:**
* Feature extraction: Lexicon-based transforms (LIWC)
* Processing: TF-IDF
* Classifier (one of): Naive Bayes, Logistic Regression, SVMs

**Approach3 :**
* Preprocessing: word embeddings (word2vec, GloVe)
  * Advantage: capturing some semantic similarity (e.g., "frightened" and "scared")
* Feature extraction: averaging, semantic pooling
* Classifier (one of): Naive Bayes, Logistic Regression, SVMs

**Approach 4:**
* Preprocessing: word embeddings (word2vec, GloVe)
* Classifier (one of): CNNs, RNNs
  * Advantage: $ \text{pattern of words} \xrightarrow{\text{signal}} \text{class} $
    * Understands context (e.g, negation)
  * Drawback: hardware, time for hyperparameter tuning, opaque

**Approach 5:**
* Embeddings: word2vec, Spacy, SBERT, Transformer ...
* Dimensionality Reduction: UMAP, PCA ...
* Clustering: k-Means, HDBSCAN, BIRCH ...
* Tokenizer: CountVectorize, POS ...
* Weighting Scheme: c-TF-IDF, c-TF-IDF BM25, c-TF-IDF+Normalization ...
* Representation Tuning: GPT/T5, KeyBERT ...
  * Advantage: fast, performant

**Approach 6:**
* Classifier (one of): finetuning pre-trained transformers (BERT and alike)
  * Advantage: best-in-class mixture of Global semantics and contextual nuances
  * Drawback: hardware, opaque
  * Comment: fine-tuned DistilBERT model (van Genugten & Schacter, 2024) with Sap et al.’s (2020) method for quantifying episodic vs semantic details


We'll explora some modeling techniques below.

In [ ]:
# @title Load the dataset
url = 'https://raw.githubusercontent.com/pinchunc/NMA_DL_SentimentAnalysis/main/Hippocorpus_data/hippoCorpusV2.csv'
df_dataset = pd.read_csv(url)
df_dataset.columns

In [ ]:
# @title Looking at the first row of data
for col, value in df_dataset.iloc[0].items():
  print(f'{col}: {value}')

## EDA on Metadata + Text Length

In [ ]:
# @title Text length values distribution & add variables to metadata
df_dataset["story_len"]   = df_dataset["story"].str.split().str.len()
df_dataset["summary_len"] = df_dataset["summary"].str.split().str.len()
sns.histplot(df_dataset["story_len"], kde=True); plt.title("Story word counts"), plt.show()
sns.histplot(df_dataset["summary_len"], kde=True); plt.title("Summary word counts"), plt.show()

In [ ]:
# @title Numeric variables values distribution by memory type
def plot_distributions(
    df,
    num_cols,
    rating_cols,
    cat_cols,
    hue_by_col=None,
    num_figsize=(6, 3),
    cat_figsize=(6, 3),
    rating_figsize=(5, 3),
    num_bins="auto",
):
    """
    Plot numeric, Likert-style rating, and categorical distributions split by a hue column.

    Parameters
    ----------
    df : pandas.DataFrame
        Your dataset.
    num_cols : list[str]
        Continuous / semi-continuous numeric columns to plot with `sns.histplot`.
    rating_cols : list[str]
        1-to-5 Likert-style rating columns to plot with `sns.countplot`.
    cat_cols : list[str]
        Other categorical columns to plot with `sns.countplot`.
    hue_by_col : str, default None
        Column used to color / split the distributions.
    num_figsize, cat_figsize, rating_figsize : tuple[float, float]
        Figure sizes for each section.
    num_bins : int | str, default "auto"
        Histogram bin specification passed to `sns.histplot`.
    """
    # ------------------------------------------------------------------
    # Numeric variables
    # ------------------------------------------------------------------
    for c in num_cols:
        plt.figure(figsize=num_figsize)
        ax = sns.histplot(
            data=df,
            x=c,
            hue=hue_by_col,
            element="step",
            stat="density",
            common_norm=False,
            kde=True,
            bins=num_bins,
        )
        plt.title(f"{c} distribution by {hue_by_col}")
        plt.tight_layout()
        plt.show()

    # ------------------------------------------------------------------
    # Likert-style ratings (assumed 1-5)
    # ------------------------------------------------------------------
    order_floats = [float(k) for k in range(1, 6)]
    for c in rating_cols:
        plt.figure(figsize=rating_figsize)
        ax = sns.countplot(
            data=df,
            x=c,
            hue=hue_by_col,
            order=order_floats,
            dodge=True,
        )
        plt.title(f"{c} distribution by {hue_by_col}")
        plt.xlabel(c)
        plt.ylabel("count")
        plt.legend(title=hue_by_col)
        plt.tight_layout()
        plt.show()

    # ------------------------------------------------------------------
    # Other categorical variables
    # ------------------------------------------------------------------
    for c in cat_cols:
        plt.figure(figsize=cat_figsize)
        ax = sns.countplot(
            data=df,
            y=c,
            hue=hue_by_col,
            order=df[c].value_counts().index,
            dodge=True,
        )
        plt.title(f"{c} by {hue_by_col}")
        plt.xlabel("count")
        plt.ylabel(c)
        plt.legend(title=hue_by_col)
        plt.tight_layout()
        plt.show()


num_cols    = ["WorkTimeInSeconds", "annotatorAge", "logTimeSinceEvent", "story_len", "summary_len"]
rating_cols = ["importance", "similarity", "frequency", "stressful", "distracted", "draining"]
cat_cols    = ["annotatorGender", "annotatorRace", "memType"]

plot_distributions(df_dataset, num_cols, rating_cols, cat_cols)
plot_distributions(df_dataset, num_cols, rating_cols, cat_cols, hue_by_col="memType")
plot_distributions(df_dataset, num_cols, rating_cols, cat_cols, hue_by_col="stressful")

In [ ]:
# @title Correlation between numeric variables
all_num_cols = num_cols + rating_cols
plt.figure(figsize=(10, 8))     # new, blank canvas
corr = df_dataset[all_num_cols].corr(numeric_only=True)
sns.heatmap(corr,
            annot=True,
            fmt=".2f",
            cmap="coolwarm",
            square=True,
            cbar_kws={"shrink": .8})
plt.title("Correlation matrix of numeric variables")
plt.tight_layout()
plt.show()

In [ ]:
# @title Difference in history length over story type
sns.boxplot(data=df_dataset, x="memType", y="story_len")
stats.ttest_ind(df_dataset[df_dataset.memType=="recalled"]["story_len"],
                df_dataset[df_dataset.memType=="imagined"]["story_len"])

---
# Step 4: hypotheses
Since humans can easily distinguish different emotions from text data, a DL model should also be able to do so. 


### Question 1: Can we predict the stress level of writing the story based on how the person told the story (i.e., vocabulary frequency)?

#### **Hypothesis:**

* **Null Hypothesis (H₀):** Vocabulary frequency does not contain enough information to predict the self-reported stress level.
* **Alternative Hypothesis (H₁):** Vocabulary frequency contains enough information to predict the self-reported stress level above chance.

#### **Mathematical Form:**

Let:

* $\mathbf{x}_i \in \mathbb{R}^d$ be the vector of word frequencies for story $i$
* $y_i \in \{1, 2, 3, 4, 5\}$ be the stress level
* $f: \mathbb{R}^d \to \{1,\dots,5\}$ be a classifier trained on $(\mathbf{x}_i, y_i)$

Then:

* **H₀:** $\mathbb{P}(f(\mathbf{x}) = y) \leq \mathbb{P}_{\text{chance}}(y) = 0.2$
* **H₁:** $\mathbb{P}(f(\mathbf{x}) = y) > 0.2$

We will use **cross-validated accuracy or F1-score** to test this empirically.

---

### Question 2: Can we predict the type of memory modality (recollection, imagination, etc.) from word frequencies?

#### **Hypothesis:**

* **H₀:** Word frequencies are not predictive of memory modality type.
* **H₁:** Word frequencies are predictive of memory modality type.

#### **Mathematical Form:**

Let:

* $z_i \in \{\text{recollection, remembered, summary, imaginative}\}$ be the memory type label

Then:

* **H₀:** $\mathbb{P}(f(\mathbf{x}) = z) \leq \mathbb{P}_{\text{chance}}(z)$
* **H₁:** $\mathbb{P}(f(\mathbf{x}) = z) > \mathbb{P}_{\text{chance}}(z)$

---

### Question 3: Does time since the event correlate with reported stress level?

#### **Hypothesis:**

* **H₀:** There is no correlation between time since the event and reported stress.
* **H₁:** There is a non-zero correlation between time since the event and reported stress.

#### **Mathematical Form:**

Let:

* $t_i \in \mathbb{R}$: time since the event (possibly log-transformed)
* $y_i \in \{1,\dots,5\}$: stress level

Then:

* **H₀:** $\text{Corr}(t, y) = 0$
* **H₁:** $\text{Corr}(t, y) \neq 0$

We could use **Pearson** or **Spearman correlation**, depending on whether the assumptions of linearity hold.

---

### Question 4:** Does including semantic information (e.g., from embeddings) significantly improve classification?

#### **Hypothesis:**

* **H₀:** Including semantic embeddings (e.g., BERT, GPT) does not significantly improve classification over frequency-only models.
* **H₁:** Including semantic embeddings significantly improves classification over frequency-only models.

#### **Mathematical Form:**

Let:

* $\mathbf{x}_{\text{freq}}$: word frequency features
* $\mathbf{x}_{\text{sem}}$: semantic embeddings
* $A_{\text{freq}}$: accuracy with only frequency
* $A_{\text{freq+sem}}$: accuracy with frequency + semantics

Then:

* **H₀:** $A_{\text{freq+sem}} - A_{\text{freq}} \leq 0$
* **H₁:** $A_{\text{freq+sem}} - A_{\text{freq}} > 0$

We. will use a **paired t-test or permutation test** across cross-validation folds to assess statistical significance.

---

### Question 5:** Are the beginning, middle, or end of the story more informative for classification?

#### **Hypothesis:**

* **H₀:** All story segments (beginning, middle, end) are equally informative for classification.
* **H₁:** Some segments of the story are significantly more informative than others for classification.

#### **Mathematical Form:**

Let:

* $A_{\text{seg}} \in \{A_{\text{beg}}, A_{\text{mid}}, A_{\text{end}}\}$ be classification accuracy per segment

Then:

* **H₀:** $A_{\text{beg}} = A_{\text{mid}} = A_{\text{end}}$
* **H₁:** $\exists i \neq j \text{ such that } A_i \neq A_j$

We. will use **ANOVA** or **post-hoc pairwise t-tests** on classification scores.


In [ ]:
# Load the dataset
url = 'https://raw.githubusercontent.com/pinchunc/NMA_DL_SentimentAnalysis/main/Hippocorpus_data/hippoCorpusV2.csv'
df = pd.read_csv(url)
df = df[['story', 'stressful']]
df.head()

In [ ]:
# Split the data into train and test
X = df['story'].values       # Text data (input)
y = df['stressful'].values   # Labels (target)

# Split data
x_train_text, x_test_text, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) # stratify helps preserve that imbalance proportionally in both the training and test sets.


In [ ]:
# Exploratory Data Analisys (EDA).
for s, l in zip(x_train_text[:5], y_train[:5]):
  print('{}: {}'.format(l, s))

In [ ]:
# List comprehension to tokenize every sentence in the text
tokenizer = get_tokenizer("basic_english")

x_train_token = [tokenizer(s) for s in tqdm(x_train_text)]
x_test_token = [tokenizer(s) for s in tqdm(x_test_text)]

# Observe the result of the tokenization of the first story
print('Before Tokenize: ', x_train_text[0])
print('After Tokenize: ', tokenizer(x_train_text[0]))

In [ ]:
# Count how many different words are present in our dataset
words = Counter()
for s in x_train_token:
  for w in s:
    words[w] += 1

sorted_words = list(words.keys())
sorted_words.sort(key=lambda w: words[w], reverse=True)
print(f"Number of different Tokens in our Dataset: {len(sorted_words)}")
print(sorted_words[:100])

In [ ]:
# Plot the distribution of different tokens
count_occurences = sum(words.values())

accumulated = 0
counter = 0

while accumulated < count_occurences * 0.8:
  accumulated += words[sorted_words[counter]]
  counter += 1

print(f"The {counter * 100 / len(words)}% most common words "
      f"account for the {accumulated * 100 / count_occurences}% of the occurrences")

In [ ]:
plt.bar(range(100), [words[w] for w in sorted_words[:100]])
plt.show()